### 01 - Instalação (Bibliotecas)

In [ ]:
!pip install torch torchvision torchaudio
!pip install openai-whisper

### 02 - Recursos (Importações)

In [ ]:
import csv

import torch
import torch.nn as nn
import torch.optim as optim

### 03 - Funções

In [ ]:
def read_csv(filename):
  try:
    with open(filename, 'r', encoding='UTF-8') as file:
      reader = csv.reader(file)

      next(reader)

      return [
        (row[1], int(row[5])) for row in reader
      ] # (Comentário, Classificação).
  except Exception as error:
    print(f"Erro ao carregar o arquivo CSV! Saída: {error}")

    exit()

def tokenize(sentence):
  return sentence.lower().split()

def encode(sentence, vocabulary, default):
  return [vocabulary.get(token, default) for token in tokenize(sentence)] # Tokens não incluídos no vocabulário são trocados pelo índice de tokens desconhecidos ("UNK_TOKEN").

def pad_sequence(sequence, max_length):
  return [0] * (max_length - len(sequence)) + sequence

def predict(sentence, vocabulary, default, model, max_length):
  model.eval()

  sequence = encode(sentence, vocabulary, default)
  sequence = pad_sequence(sequence, max_length)

  sequence = torch.tensor([sequence])

  # print(f"\nTensor ({sentence}):\n\n{sequence}") # Utilize essa linha para ver o efeito da função "encode" em tokens desconhecidos, onde o índice aplicado a eles é "1" ("UNK_TOKEN").

  with torch.no_grad():
    output = model(sequence).item()

  return f"{'Ofensivo':12} ({output:.15f})" if output > 0.5 else f"{'Não Ofensivo':12} ({output:.15f})" # Contexto do dataset oposto ao do dicionário desenvolvido em aula.

### 04 - Classe

In [ ]:
class RNN_Sentiment(nn.Module):
  def __init__(self, vocabulary_size, embedding_dim, hidden_dim):
    super(RNN_Sentiment, self).__init__()

    self.embedding = nn.Embedding(vocabulary_size, embedding_dim, padding_idx=0)

    self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

    self.fc = nn.Linear(hidden_dim, 1)

    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.embedding(x)

    _, hidden = self.rnn(x)

    output = self.fc(hidden[-1])

    return self.sigmoid(output)

### 05 - Implementação

In [ ]:
sentences = [
  (commentary, label) for (commentary, label) in read_csv('./Data/HateBR.csv')
]

UNK_TOKEN = 1

vocabulary = { "<UNK>": UNK_TOKEN }

for sentence, _ in sentences:
 for token in tokenize(sentence):
  if (token not in vocabulary):
   vocabulary[token] = len(vocabulary) + 1

vocabulary_size = len(vocabulary) + 1

encoded_data = [(encode(sentence, vocabulary, UNK_TOKEN), label) for sentence, label in sentences]

max_length = max(len(sequence) for sequence, _ in encoded_data)

x = torch.tensor([pad_sequence(sequence, max_length) for sequence, _ in encoded_data])

y = torch.tensor([label for _, label in encoded_data], dtype=torch.float32)

print("Tensores:\n")

print(x)

embedding_dim = 10

hidden_dim = 16

model = RNN_Sentiment(
 vocabulary_size,
 embedding_dim,
 hidden_dim
)

### 06 - Treinamento

In [ ]:
criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 100

for epoch in range(0, epochs):
  model.train()

  outputs = model(x).squeeze()

  loss = criterion(outputs, y)

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  if ((epoch + 1) % 10 == 0):
    print(
      f"| Época: {(epoch + 1):10}",
      f"| Perda: {loss.item():10.5f} |"
    )

### 07 - Aplicação

In [ ]:
print("Testes Aplicados:")

print(
  f"\n| Frase: {'Habilidade incrível':50}",
  f"| Resultado: {predict('Habilidade incrível', vocabulary, UNK_TOKEN, model, max_length)} |"
)

print(
  f"\n| Frase: {'País de merda':50}",
  f"| Resultado: {predict('País de merda', vocabulary, UNK_TOKEN, model, max_length)} |"
)

# Demonstrando que tokens não incluídos no vocabulário são trocados pelo índice de tokens desconhecidos ("UNK_TOKEN").

print(
  f"\n| Frase: {'Lorem Ipsum Dolor é incrível':50}",
  f"| Resultado: {predict('Lorem Ipsum Dolor é incrível', vocabulary, UNK_TOKEN, model, max_length)} |"
)

print(
  f"\n| Frase: {'Lorem Ipsum Dolor é uma merda':50}",
  f"| Resultado: {predict('Lorem Ipsum Dolor é uma merda', vocabulary, UNK_TOKEN, model, max_length)} |"
)